# LightGBM Experiments

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV

from sklearn.feature_selection import SelectFromModel
from sklearn.inspection import permutation_importance
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, f1_score, accuracy_score, balanced_accuracy_score, ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_class_weight

import lightgbm as lgb
from lightgbm import LGBMClassifier
from imblearn.over_sampling import RandomOverSampler

In [2]:
# import sys
# !{sys.executable} -m pip install pyarrow
# !{sys.executable} -m pip install --upgrade pyarrow pandas

## Dataloader

In [4]:
# load train data
X_train_raw = pd.read_parquet("../data/model_raw/X_train.parquet")
y_train_raw = pd.read_parquet("../data/model_raw/Y_train.parquet")
y_train_raw = np.where(y_train_raw['depression_severity'] > 0, 1, 0) # create binary labels

# upsample train data - do what you need to do
ros = RandomOverSampler(sampling_strategy='not majority', random_state=7)
X_train_balanced, y_train_balanced = ros.fit_resample(X_train_raw, y_train_raw)

# convert X and Y to np array
X_train = np.array(X_train_balanced)
y_train = np.array(y_train_balanced)

# load validation data
X_val = pd.read_parquet("../data/model_raw/X_val.parquet")
y_val = pd.read_parquet("../data/model_raw/Y_val.parquet")['depression_severity']
X_val = np.array(X_val)
y_val = np.array(np.where(y_val > 0, 1, 0)) # np array of binary labels

# load test data
X_test = pd.read_parquet("../data/model_raw/X_test.parquet")
y_test = pd.read_parquet("../data/model_raw/Y_test.parquet")['depression_severity']
X_test = np.array(X_test)
y_test = np.array(np.where(y_test > 0, 1, 0)) # np array of binary labels

## Baseline Model

In [8]:
lgbm_base = LGBMClassifier(
    objective='binary',
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgbm_base.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=-1)]
)

y_pred_base  = lgbm_base.predict(X_test)
y_proba_base = lgbm_base.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_base, target_names=['None/Minimal', 'Mild/Severe']))
print(f'ROC-AUC : {roc_auc_score(y_test, y_proba_base):.4f}')

              precision    recall  f1-score   support

None/Minimal       0.84      0.84      0.84       776
 Mild/Severe       0.48      0.50      0.49       238

    accuracy                           0.76      1014
   macro avg       0.66      0.67      0.66      1014
weighted avg       0.76      0.76      0.76      1014

ROC-AUC : 0.7385


/home/jovyan/myenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jovyan/myenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Hyperparameter Tuning

In [ ]:
param_dist = {
    'n_estimators'      : [200, 400],
    'learning_rate'     : [0.001, 0.01, 0.05],
    'num_leaves'        : [10, 20, 30],
    'max_depth'         : [-1, 5, 10, 15],
    # 'min_child_samples' : [10, 20, 30, 50],
    # 'subsample'         : [0.7, 0.8, 1.0],
    # 'colsample_bytree'  : [0.7, 0.8, 1.0],
    # 'reg_alpha'         : [0, 0.01, 0.1, 0.5],
    # 'reg_lambda'        : [0, 0.01, 0.1, 1.0],
}

lgbm_tuned = LGBMClassifier(objective='binary', random_state=42, n_jobs=-1, verbose=-1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    lgbm_tuned,
    param_distributions=param_dist,
    n_iter=50,
    scoring='roc_auc',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
search.fit(X_train, y_train)

print('Best params:', search.best_params_)
print('Best CV ROC-AUC:', round(search.best_score_, 4))

Fitting 5 folds for each of 50 candidates, totalling 250 fits


## Best Set of Parameters

In [ ]:
best_model = search.best_estimator_

y_pred_val  = best_model.predict(X_val)
y_proba_val = best_model.predict_proba(X_val)[:, 1]

print('Best Model - Validation')
print(classification_report(y_val, y_pred_val, target_names=['None/Minimal', 'Mild→Severe']))
print(f'ROC-AUC : {roc_auc_score(y_val, y_proba_val):.4f}')

y_pred_test  = best_model.predict(X_test)
y_proba_test = best_model.predict_proba(X_test)[:, 1]

print('\n Best Model - ')
print(classification_report(y_test, y_pred_test, target_names=['None/Minimal', 'Mild→Severe']))
print(f'ROC-AUC : {roc_auc_score(y_test, y_proba_test):.4f}')

## Evaluation & Fairness

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (preds, labels, title) in zip(
    axes,
    [
        (y_pred_val,  y_val,  'Validation Set'),
        (y_pred_test, y_test, 'Test Set'),
    ]
):
    cm = confusion_matrix(labels, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=['None/Minimal', 'Mild→Severe'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'LightGBM — {title}')

plt.tight_layout()
plt.savefig('lightgbm_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for proba, labels, name, ls in [
    (y_proba_val,  y_val_bin,  'Validation', '--'),
    (y_proba_test, y_test_bin, 'Test',       '-'),
]:
    fpr, tpr, _ = roc_curve(labels, proba)
    auc = roc_auc_score(labels, proba)
    ax.plot(fpr, tpr, ls=ls, lw=2, label=f'{name} AUC = {auc:.3f}')

ax.plot([0,1],[0,1], 'k:', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('LightGBM ROC Curve\n(Grouped: Mild→Severe vs None/Minimal)')
ax.legend()
plt.tight_layout()
plt.savefig('lightgbm_roc.png', dpi=150, bbox_inches='tight')
plt.show()